# 01: Exploratory data analysis (train set only)

Goal: understand the cleaned data and collect ideas for M4 feature engineering.
Use the **train** split only. The test set stays unseen until M6.

Run `make split` first if `load_split` says the file is missing.

In [1]:
%load_ext autoreload
%autoreload 2

import altair as alt
import altair_ally as aly
import pandas as pd
from sklearn.metrics import roc_auc_score

from credit_risk.config import default_config_path, load_config
from credit_risk.data import (
    BILL_AMTS,
    PAY_AMTS,
    PAY_STATUS,
    RAW_TARGET,
    TARGET,
    load_raw,
    load_split,
)

# VegaFusion runs chart transforms (density, aggregates) in Python and embeds only
# the results, so the notebook doesn't store all ~24k training rows per chart
alt.data_transformers.enable("vegafusion")

CONTINUOUS = ["LIMIT_BAL", "AGE", *BILL_AMTS, *PAY_AMTS]
DISCRETE = ["SEX", "EDUCATION", "MARRIAGE", *PAY_STATUS]

train = load_split("train")
train.shape

(24000, 25)

## Key takeaways

- **Imbalanced target:** 22.1% of training clients default, so accuracy is a misleading metric (section 1).
- **Repayment history carries the most signal.** The latest month (PAY_1) is the strongest single column, and counts across all six months are stronger still: default rate climbs from 12% for clients never late to 70% for clients late every month (section 3b).
- **Payment behaviour matters more than bill size.** Clients who paid nothing toward a positive bill default at 40%, while clients who paid more than their bill default at 14% (section 4a). Bill amounts look similar for both classes (sections 2 and 4).
- **Lower credit limits mean higher risk:** 32% default at limits up to NT$50k against 15% above NT$200k (section 2).
- **Bill amounts are highly correlated with each other** (0.80 to 0.95), so they're largely redundant for modelling (section 5).
- **Data quirks to handle in M4:** undocumented codes (already merged in cleaning), PAY code 1 missing after the first month, and skewed amounts with negative bills.

## 1. Class balance

What share of clients default? Why does that make accuracy a poor metric here?

In [2]:
props = train[TARGET].value_counts(normalize=True).reset_index()

alt.Chart(props).mark_bar().encode(
    y=alt.Y("default:N", title="Default status", sort="-x"),
    x=alt.X("proportion:Q", title="Share of clients", axis=alt.Axis(format="%")),
    color=alt.Color("default:N", legend=None),
    tooltip=[
        alt.Tooltip("default:N", title="Default status"),
        alt.Tooltip("proportion:Q", title="Share", format=".1%"),
    ],
).properties(title="Target class imbalance in training data")

alt.Chart(...)

**Findings**

- 5,309 of 24,000 training clients (22.1%) default; 18,691 (77.9%) don't.
- A model that always predicts "no default" would score 77.9% accuracy while catching no defaulters, so accuracy hides the errors that matter here.
- Better metrics for this problem: ROC AUC and precision-recall AUC for ranking, plus recall and precision on the default class at a chosen threshold. The train/test split is stratified, so both sets keep this 22% rate.

## 2. Continuous features by class

Do defaulters and non-defaulters differ in credit limit, age, bills or payments? Each class is normalized to 100%, so bar heights compare shapes, not counts. BILL_AMT and PAY_AMT use a signed log10 scale: 0 means zero, 3 about 1,000, 5 about 100,000, and negative values are credit balances.

In [3]:
def class_hist(data, col, log=False, maxbins=30, width=220, height=120):
    """Histogram of `col` with one overlaid bar series per class.

    Bar height is the share of that class's clients in each bin, so both classes
    sum to 100% and the 78/22 imbalance doesn't hide the default class.
    log=True bins sign(x) * log10(1 + |x|), which spreads out skewed amounts
    and keeps negative values (credit balances in BILL_AMT).
    """
    value = f"datum['{col}']"
    if log:
        value = f"({value} < 0 ? -1 : 1) * log(1 + abs({value})) / log(10)"
    return (
        alt.Chart(data[[col, TARGET]])
        .transform_calculate(
            value=value,
            status=f"datum['{TARGET}'] == 1 ? 'Default' : 'No default'",
        )
        .transform_bin(["bin_start", "bin_end"], "value", bin=alt.Bin(maxbins=maxbins))
        .transform_aggregate(n="count()", groupby=["status", "bin_start", "bin_end"])
        .transform_joinaggregate(class_n="sum(n)", groupby=["status"])
        .transform_calculate(share="datum.n / datum.class_n")
        .mark_bar(opacity=0.5)
        .encode(
            x=alt.X(
                "bin_start:Q",
                bin="binned",
                title=f"{col} (signed log10)" if log else col,
                axis=alt.Axis(format="~s"),
            ),
            x2="bin_end:Q",
            y=alt.Y("share:Q", title="Share of class", axis=alt.Axis(format="%"), stack=None),
            color=alt.Color("status:N", title="Default status", sort=["No default", "Default"]),
            tooltip=[
                alt.Tooltip("status:N", title="Status"),
                alt.Tooltip("bin_start:Q", title="From", format=",.2~f"),
                alt.Tooltip("bin_end:Q", title="To", format=",.2~f"),
                alt.Tooltip("share:Q", title="Share of class", format=".1%"),
                alt.Tooltip("n:Q", title="Clients", format=","),
            ],
        )
        .properties(width=width, height=height)
    )

In [4]:
class_hist(train, "LIMIT_BAL", maxbins=40, width=500, height=200).properties(
    title="Credit limit by default status"
)

alt.Chart(...)

In [5]:
alt.concat(
    class_hist(train, "LIMIT_BAL"),
    class_hist(train, "AGE"),
    *[class_hist(train, col, log=True) for col in [*BILL_AMTS, *PAY_AMTS]],
    columns=3,
).properties(title="Continuous features by default status (share of each class)")

alt.ConcatChart(...)

**Findings**

- **Credit limit:** defaulters have lower limits (median NT$90k against NT$150k). The default rate falls from 32% at limits up to NT$50k to 26% at NT$50k to 100k, 20% at NT$100k to 200k and 15% above NT$200k. A lower limit likely reflects the bank's own risk assessment, so this is a strong but partly second-hand signal.
- **Age:** both classes have a median age of 34, but the default rate is U-shaped: 27% for ages 21 to 25, about 20% for 26 to 40, and 24 to 27% above 40. A linear term would miss this shape.
- **Bill amounts:** the two classes have similar shapes in every month. Bill size alone says little about default.
- **Payment amounts:** defaulters are about twice as likely to pay nothing in a month (29% against 14% for PAY_AMT1), and the gap holds in all six months. Among clients who did pay, defaulters pay somewhat less (median NT$2,500 against NT$3,000 for PAY_AMT1).

## 3. Categorical features

Counts per category mostly repeat the class imbalance, so the second chart shows the default rate per category instead. The dashed line is the overall rate; faded bars have fewer than 100 clients. PAY codes 3 to 8 are grouped as "3+" in the chart only.

In [6]:
# aly colours by a categorical, so cast the 0/1 target to string
eda = train.assign(**{TARGET: train[TARGET].astype(str)})

In [7]:
aly.dist(eda[[*DISCRETE, TARGET]].astype(str), dtype="object", color=TARGET)

alt.ConcatChart(...)

In [8]:
CATEGORY_ORDER = ["-2", "-1", "0", "1", "2", "3", "4", "3+"]

long = train.melt(id_vars=TARGET, value_vars=DISCRETE, var_name="variable", value_name="category")

# Group 3+ months late into one bucket for PAY_* only; codes 4-8 each have <100 clients
is_late = long["variable"].isin(PAY_STATUS) & (long["category"] >= 3)
long["category"] = long["category"].astype(str).where(~is_late, "3+")

rates = (
    long.groupby(["variable", "category"])[TARGET].agg(default_rate="mean", n="size").reset_index()
)
base_rate = train[TARGET].mean()

bars = (
    alt.Chart()
    .mark_bar()
    .encode(
        x=alt.X("category:N", title=None, sort=CATEGORY_ORDER),
        y=alt.Y("default_rate:Q", title="Default rate", axis=alt.Axis(format="%")),
        opacity=alt.condition("datum.n >= 100", alt.value(1), alt.value(0.4)),
        tooltip=[
            alt.Tooltip("category:N", title="Category"),
            alt.Tooltip("default_rate:Q", title="Default rate", format=".1%"),
            alt.Tooltip("n:Q", title="Clients", format=","),
        ],
    )
)
overall = alt.Chart().mark_rule(color="gray", strokeDash=[4, 4]).encode(y=alt.datum(base_rate))

alt.layer(bars, overall, data=rates).properties(width=180, height=140).facet(
    facet=alt.Facet("variable:N", title=None, sort=DISCRETE), columns=3
).resolve_scale(x="independent").properties(
    title=f"Default rate by category (dashed line = overall {base_rate:.1%})"
)

alt.FacetChart(...)

**Findings**

- **PAY_x** separates the classes far more than any demographic column. In PAY_1, clients who paid the minimum or better (codes -2 to 0) default at 13 to 17%, while clients 2 or more months late default at about 70%.
- **SEX:** men (code 1) default at 24.5% and women (code 2) at 20.6%. The gap is real but small, and using sex in credit decisions raises fairness and legal concerns, so treat it with care in M4.
- **EDUCATION and MARRIAGE:** small differences between the main groups (19 to 25% and 21 to 23%). See 3a for the merged codes.

### 3a. The cleaned categories

EDUCATION (4 = other/unknown after cleaning) and MARRIAGE (3 = other).
Compare default rates per group. Does merging the undocumented codes look reasonable?

In [9]:
raw = load_raw(load_config(default_config_path()))
raw_train = raw[raw["ID"].isin(train["ID"])]  # training clients only, before cleaning

for col in ["EDUCATION", "MARRIAGE"]:
    before = raw_train.groupby(col)[RAW_TARGET].agg(default_rate="mean", n="size")
    after = train.groupby(col)[TARGET].agg(default_rate="mean", n="size")
    display(pd.concat({"raw codes": before, "after cleaning": after}, axis=1).round(3))

raw codes        after cleaning         
          default_rate      n   default_rate        n
EDUCATION                                            
0                0.000     11            NaN      NaN
1                0.194   8487          0.194   8487.0
2                0.238  11228          0.238  11228.0
3                0.248   3914          0.248   3914.0
4                0.069    102          0.067    371.0
5                0.055    218            NaN      NaN
6                0.150     40            NaN      NaN

raw codes        after cleaning         
         default_rate      n   default_rate        n
MARRIAGE                                            
0               0.116     43            NaN      NaN
1               0.234  10889          0.234  10889.0
2               0.210  12826          0.210  12826.0
3               0.252    242          0.232    285.0

**Findings**

- EDUCATION: the undocumented codes 0, 5 and 6 (269 clients) default at 0 to 15%, close to code 4 (6.9%) and well below codes 1 to 3 (19 to 25%). Merging them into 4 looks reasonable. The merged group defaults at 6.7%, much lower than the others, so it may carry signal of its own.
- MARRIAGE: code 0 (43 clients) defaults at 11.6% against 25.2% for code 3. The rates differ, but 43 clients is too few to trust the gap, and the merge moves code 3's rate only from 25.2% to 23.2%.
- MARRIAGE differs little by group overall (21 to 23%), so expect it to be a weak predictor.

### 3b. Repayment status (PAY_1 to PAY_6)

-2 = no card use, -1 = paid in full, 0 = paid the minimum, 1 to 8 = months late.
How does the default rate change across these values? Is PAY_1 (most recent) more informative than PAY_6?

Note: code 1 (one month late) is common in PAY_1 but has 22 or fewer clients in PAY_2 to PAY_5 and none in PAY_6. The cause is undocumented; handle it in M4.

In [10]:
# Helper used here and in section 4.
# `groups` is a Series with one label per client (for example "0 months late").
def rate_chart(groups, title, width=400):
    """Bar chart of the default rate in each group, with the overall rate as a dashed line."""
    # Mean of a 0/1 column = share of 1s, so mean() gives the default rate per group
    rates = train[TARGET].groupby(groups, observed=True).agg(default_rate="mean", n="size")
    rates = rates.rename_axis("group").reset_index().astype({"group": str})
    bars = (
        alt.Chart(rates)
        .mark_bar()
        .encode(
            x=alt.X("group:N", title=None, sort=rates["group"].tolist()),
            y=alt.Y("default_rate:Q", title="Default rate", axis=alt.Axis(format="%")),
            tooltip=[
                alt.Tooltip("group:N", title="Group"),
                alt.Tooltip("default_rate:Q", title="Default rate", format=".1%"),
                alt.Tooltip("n:Q", title="Clients", format=","),
            ],
        )
    )
    overall = alt.Chart().mark_rule(color="gray", strokeDash=[4, 4]).encode(y=alt.datum(base_rate))
    return (bars + overall).properties(title=title, width=width, height=180)

**Step 1: summarize six months into one number per client**

Each client has six repayment codes (PAY_1 to PAY_6). Two ways to squash them into one column:

- `max_delay`: the worst code across the six months. A client with codes 0, 2, 0, -1, 0, 0 gets 2 (two months behind at some point).
- `months_late`: how many of the six months had a code of 1 or more. The same client gets 1.

The table shows a few clients so you can check the two columns against the raw codes.

In [11]:
pay_summary = pd.DataFrame(
    {
        # Worst (highest) code in each row; axis=1 means across columns, per client
        "max_delay": train[PAY_STATUS].max(axis=1),
        # True/False per month (late or not), then sum counts the Trues per client
        "months_late": (train[PAY_STATUS] >= 1).sum(axis=1),
    }
)

# A few clients with at least one late month, raw codes next to the summaries
pd.concat([train[PAY_STATUS], pay_summary], axis=1)[pay_summary["months_late"] > 0].head()

,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,max_delay,months_late
0,2,2,-1,-1,-2,-2,2,2
1,-1,2,0,0,0,2,2,2
6,0,0,2,0,0,0,2,1
7,0,0,2,0,0,-1,2,1
8,-1,-1,-1,-1,-1,2,2,1


**Step 2: score each feature with AUC**

AUC (area under the ROC curve) asks: *if you pick one defaulter and one non-defaulter at random, how often does this feature give the defaulter the higher value?*

- 1.0 = the defaulter is higher every time (perfect separation)
- 0.5 = a coin flip (no signal)
- below 0.5 = the relationship runs the other way

Here each feature is used on its own as the "score", with no model. `roc_auc_score(y_true, score)` from scikit-learn does the calculation. AUC isn't affected by the 78/22 class imbalance, and it's a common metric for the final model too, so these numbers are a baseline to beat.

In [12]:
# Every feature to score: the six monthly columns plus the two summaries
features = pd.concat([train[PAY_STATUS], pay_summary], axis=1)

auc_scores = {}
for col in features.columns:
    # y_true = actual default (0/1); score = the feature's value for each client
    auc_scores[col] = roc_auc_score(train[TARGET], features[col])

auc = pd.Series(auc_scores).rename_axis("feature").reset_index(name="auc")
auc.round(3)

,feature,auc
0,PAY_1,0.689
1,PAY_2,0.635
2,PAY_3,0.621
3,PAY_4,0.606
4,PAY_5,0.599
5,PAY_6,0.588
6,max_delay,0.705
7,months_late,0.724


**Step 3: chart the results**

Left: the AUC of each feature (the axis starts at 0.5, the no-signal line). Right: default rate for each value of `months_late`, with the overall 22% rate as a dashed line.

In [13]:
auc_chart = (
    alt.Chart(auc)
    .mark_bar(clip=True)  # axis starts at 0.5, so clip bars at the edge
    .encode(
        y=alt.Y("feature:N", title=None, sort=auc["feature"].tolist()),
        x=alt.X("auc:Q", title="AUC (single feature)", scale=alt.Scale(domain=[0.5, 0.75])),
        tooltip=[alt.Tooltip("feature:N"), alt.Tooltip("auc:Q", format=".3f")],
    )
    .properties(title="How well each repayment feature ranks defaulters", width=300)
)

auc_chart | rate_chart(pay_summary["months_late"], "Default rate by months late (of 6)")

alt.HConcatChart(...)

**Findings**

- The signal weakens with age: AUC falls from 0.69 for PAY_1 (September) to 0.59 for PAY_6 (April). PAY_1 is the most informative single month.
- Summaries across all six months do better than any single month: max delay 0.71, number of months late 0.72.
- Default rate rises steadily with months late, from 12% for clients never late to 70% for clients late in all six months.
- "Late" here means code 1 or higher, so it inherits the code 1 quirk in PAY_2 to PAY_6 (see the note above).
- AUC treats the codes as ordered numbers. That fits here because -2 to 8 runs roughly from least to most risky.

## 4. Bills and payments

Look at BILL_AMT and PAY_AMT against LIMIT_BAL. Try a quick version of credit utilization (BILL_AMT1 / LIMIT_BAL) and a payment ratio (PAY_AMT1 / BILL_AMT1). Watch for division by zero.

In [14]:
bills = pd.DataFrame(
    {
        # Share of the credit limit used on the latest statement; negative = credit balance
        "utilization": train["BILL_AMT1"] / train["LIMIT_BAL"],
        # Share of the latest bill paid; undefined when there's no positive bill, capped at 1
        "pay_ratio": (train["PAY_AMT1"] / train["BILL_AMT1"])
        .where(train["BILL_AMT1"] > 0)
        .clip(upper=1),
    }
)
display(bills.describe().round(3))

util_groups = pd.qcut(bills["utilization"], 10, precision=2)
pay_groups = (
    pd.cut(
        bills["pay_ratio"],
        [-0.01, 0, 0.05, 0.1, 0.25, 0.5, 0.99, 1],
        labels=["0%", "0-5%", "5-10%", "10-25%", "25-50%", "50-99%", "100%"],
    )
    .cat.add_categories("no bill")
    .fillna("no bill")
)

rate_chart(util_groups, "Default rate by utilization decile (BILL_AMT1 / LIMIT_BAL)") | rate_chart(
    pay_groups, "Default rate by share of bill paid (PAY_AMT1 / BILL_AMT1)"
)

,utilization,pay_ratio
count,24000.000,21946.000
mean,0.424,0.247
std,0.412,0.351
min,-0.552,0.000
25%,0.022,0.038
50%,0.316,0.061
75%,0.829,0.271
max,6.455,1.000


alt.HConcatChart(...)

**Findings**

- **Utilization** is U-shaped. Clients using almost none of their limit default at 20 to 25%, the rate bottoms out near 15% at 1 to 13% utilization, then rises to 29% for clients at or over their limit. Alone it ranks defaulters weakly (AUC 0.55), and the shape isn't monotonic, so bins or a tree model will use it better than a linear term.
- **Payment ratio** has a clearer signal. Clients who paid nothing toward a positive bill default at 40%, clients paying up to a quarter at about 20%, and clients paying half or more at 13 to 16%. Clients with no positive bill (9%) default at 25%.
- Division by zero: LIMIT_BAL is never 0. BILL_AMT1 is 0 or negative for 9% of clients, so the payment ratio is set to missing there and grouped as "no bill".
- Timing: PAY_AMT1 is the amount paid in September, which may be paying the August statement (BILL_AMT2). Worth trying both pairings in M4.

### 4a. Paying more than the bill

`pay_ratio` above is capped at 1, which lumps overpayers in with clients who paid exactly their bill. This check uses the uncapped ratio, and compares the two possible bill pairings for PAY_AMT1 (same month vs previous statement).

In [15]:
positive_bill = train["BILL_AMT1"] > 0
raw_ratio = (train["PAY_AMT1"] / train["BILL_AMT1"]).where(positive_bill)

paid = pd.cut(
    raw_ratio,
    [-0.01, 0, 0.999999, 1.000001, float("inf")],
    labels=["paid nothing", "paid part", "paid exactly the bill", "paid more than the bill"],
)
overpay = train[TARGET].groupby(paid, observed=True).agg(default_rate="mean", n="size")
overpay["share"] = overpay["n"] / positive_bill.sum()
display(overpay.round(3))

# Timing: does PAY_AMT1 look like it pays this month's bill or the previous statement?
pairings = {
    "PAY_AMT1 / BILL_AMT1": train["BILL_AMT1"],
    "PAY_AMT1 / BILL_AMT2": train["BILL_AMT2"],
}
pd.Series(
    {name: (train["PAY_AMT1"] >= 0.99 * bill)[bill > 0].mean() for name, bill in pairings.items()},
    name="share paying at least 99% of the bill",
).round(3)

,default_rate,n,share
paid nothing,0.398,2809,0.128
paid part,0.199,16060,0.732
paid exactly the bill,0.232,538,0.025
paid more than the bill,0.138,2539,0.116


PAY_AMT1 / BILL_AMT1    0.141
PAY_AMT1 / BILL_AMT2    0.255
Name: share paying at least 99% of the bill, dtype: float64

**Findings**

- 11.6% of clients with a positive bill paid more than it (median 2.3 times the bill). They default at 13.8%, the lowest of any group here, so capping the ratio at 1 hid a low-risk group.
- Clients who paid exactly the bill default at 23.2%, close to the overall rate. Only 2.5% of clients fall here, which suggests the same-month pairing is not how most payments work.
- 25.5% of clients paid at least 99% of the *previous* statement (BILL_AMT2), against 14.1% for the same-month bill (a share that includes the overpayers). This supports pairing PAY_AMT1 with BILL_AMT2 in M4.

## 5. Correlations

Pearson and Spearman correlations (ID dropped). Expect BILL_AMT1 to BILL_AMT6 to be highly correlated with each other.

In [16]:
aly.corr(train.drop(columns="ID"))

alt.ConcatChart(...)

**Findings**

- **Strongest links with default** are the PAY_x columns (Pearson 0.32 for PAY_1 down to 0.18 for PAY_6), followed by LIMIT_BAL and PAY_AMT1 (Spearman about -0.16 each). No single column is strongly correlated with the target, so the model will need to combine several.
- **Multicollinearity:** BILL_AMT1 to BILL_AMT6 correlate at 0.80 to 0.95 (Pearson), since a balance carries over from month to month. PAY_1 to PAY_6 correlate at 0.47 to 0.82. Summary features (for example the latest bill or an average) can replace the six raw columns without losing much.
- **Payment amounts** correlate more strongly on the rank (Spearman 0.45 to 0.55) than on the raw scale (Pearson 0.14 to 0.34), a sign that a few very large payments distort the Pearson values.
- Correlation only measures monotonic links, so it misses the U-shapes found for age and utilization.

## 6. Feature ideas for M4

Each idea is tested in M4 against the M2 baseline, which already sees the raw columns.

1. **Months late in the last six** (count of PAY_x ≥ 1): the strongest single feature found here (AUC 0.72). It captures a habit of late payment rather than one bad month.
2. **Maximum delay across six months** (max of PAY_x): how far behind a client has ever fallen (AUC 0.71), which the most recent month alone can miss.
3. **Latest repayment status with 3+ grouped** (PAY_1, codes 3 to 8 merged): the most recent month carries the most signal, and grouping removes tiny, noisy categories.
4. **Share of the previous bill paid** (PAY_AMT1 / BILL_AMT2, uncapped, with a "no bill" flag): paying nothing doubles the default rate, paying more than the bill marks a low-risk group (4a), and the previous statement looks like the bill being paid.
5. **Months with zero payment** (count of PAY_AMTx = 0): defaulters are over-represented at zero in every month, so a count summarizes that across the six months.
6. **Credit utilization in bins** (BILL_AMT1 / LIMIT_BAL): risk is U-shaped in utilization, so bins (or a tree model) fit it better than a raw ratio.

Open question for M4: PAY code 1 almost never appears after PAY_1 (section 3b), so "late" may mean something different in the earlier months.